# 本会議 議事録変換ツール 2026-03-1

---
### 使い方
1. **セル①** ライブラリ読み込み（最初に1回実行）
2. **セル②** 設定（入出力フォルダ・除去パターンなど）
3. **セル③** 変換コア関数の定義（1回実行）
4. **セル④** 実行（変換したいときに実行）
5. **セル⑤** 変換結果のプレビュー（任意）

In [ ]:
# セル① ライブラリ読み込み
import re
from pathlib import Path
from IPython.display import display, Markdown
print('✅ ライブラリ読み込み完了')


In [ ]:
# セル② 設定（ここだけ編集してください）

# 📂 入力フォルダ
# 例: INPUT_DIR = Path(r'C:\Users\yourname\Documents\gijiroku')
INPUT_DIR = Path('.')

# 📂 出力フォルダ（None = INPUT_DIR と同じ）
OUTPUT_DIR = None

OUTPUT_PREFIX = 'Cleaned_'
INPUT_EXT     = '.txt'
OUTPUT_EXT    = '.md'
ENCODINGS     = ['cp932', 'utf-8-sig', 'utf-8', 'shift_jis']

# 🧹 ノイズ除去パターン
IGNORE_PATTERNS = [
    # 開会・成立
    re.compile(r'出席議員は定足数に達しておりますので'),
    re.compile(r'議会は成立いたしました'),
    re.compile(r'これより[令平][和成].*名取市議会.*を開会いたします'),
    re.compile(r'これより本日の会議を開きます'),
    # 議事日程・署名
    re.compile(r'本日の議事日程は'),
    re.compile(r'会議録署名議員の指名を行います'),
    re.compile(r'指名いたします'),
    # 報告関連
    re.compile(r'これをもって諸般の報告を終わります'),
    re.compile(r'これをもって報告を終わります'),
    # 見出し（◯なし・空白始まりの短い区切り見出し）
    re.compile(r'^諸般の報告$'),
    re.compile(r'^一般市政報告$'),
    re.compile(r'^〔.+〕$'),  # 登壇・起立・異議なし等の〔〕行をまとめて除去
    re.compile(r'御異議なしと認めます。$'),  # 文末のみ除去（後続文がある行は残す）
    # 散会・閉会
    re.compile(r'本日はこれにて散会いたします'),
    re.compile(r'これより本日の会議を閉じます'),
    # 礼
    re.compile(r'（修　礼）'),
    # 時刻行（開会・開議・休憩・再開・散会・閉会）
    re.compile(r'^午[前後][０-９\d時分　]+[開休再散閉議会憩]'),
    # 「散　　　会」「閉　　　会」などの区切り行
    re.compile(r'^[散閉開休再][\s\u3000]+[会議憩]'),
]

print('✅ 設定完了')
print(f'   入力フォルダ: {INPUT_DIR.resolve()}')
print(f'   出力フォルダ: {(OUTPUT_DIR or INPUT_DIR).resolve()}')


In [ ]:
# セル③ 変換コア関数の定義

SPEAKER_RE = re.compile(
    r'^◯'
    r'('
    r'[^\s\u3000（]*（[^）]*）'
    r'|'
    r'[^\s\u3000]+'
    r')'
    r'(?:[\s\u3000]+(.*))?$'
)
SEPARATOR_RE = re.compile(r'^[─━ー\-]{5,}')
SCHEDULE_RE  = re.compile(r'^\s*日程第[０-９\d]+')
DATETIME_RE  = re.compile(r'^午[前後][０-９\d]+時')
SIGNATURE_RE = re.compile(r'地方自治法第１２３条第２項の規定によりここに署名する')
REDACT_RE    = re.compile(r'＿＿+')


def read_file(path):
    for enc in ENCODINGS:
        try:
            return path.read_text(encoding=enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return None


def is_noise(line):
    if SEPARATOR_RE.match(line): return True
    if '○' in line and len(line) > 20: return True
    if DATETIME_RE.match(line): return True
    # ◯で始まる発言行はIGNORE_PATTERNSを適用しない（部分一致による誤削除防止）
    if line.startswith('◯'): return False
    return any(p.search(line) for p in IGNORE_PATTERNS)


def format_speaker_line(line):
    m = SPEAKER_RE.match(line)
    if m:
        return f'**{m.group(1)}**: {(m.group(2) or "").strip()}'
    return line.replace('◯', '')


def apply_redaction(line):
    """＿が連続する箇所を（発言取消）に置き換える。"""
    return REDACT_RE.sub('（発言取消）', line)


def is_schedule_continuation(line):
    """日程見出し行の継続行（折り返し・補足括弧）かどうか判定する。"""
    if not line: return False
    if re.match(r'^(日程第|◯|─|━)', line): return False
    if re.match(r'^\d+:\s*$', line): return False
    return True


def strip_signature(lines):
    """署名ブロック（地方自治法第123条〜）以降の行を除去して返す。"""
    for i, line in enumerate(lines):
        if SIGNATURE_RE.search(line):
            return lines[:i]
    return lines


def clean_lines(lines):
    lines = strip_signature(lines)
    cleaned = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        i += 1
        if not line: continue
        if re.match(r'^\d+:\s*$', line): continue
        if is_noise(line): continue
        line = re.sub(r'^\d+:\s*', '', line)
        if SCHEDULE_RE.match(line):
            # ケース①: 「〜から」で終わる → 次行「〜まで」と結合して範囲見出しにする
            if line.rstrip().endswith('から'):
                next_raw = lines[i].strip() if i < len(lines) else ''
                next_line = re.sub(r'^\d+:\s*', '', next_raw)
                if SCHEDULE_RE.match(next_line) and next_line.rstrip().endswith('まで'):
                    line = line.strip(chr(0x3000)).strip() + '／' + next_line.strip(chr(0x3000)).strip()
                    i += 1
            # ケース②: 折り返し・補足括弧の継続行を連結する
            else:
                while i < len(lines):
                    next_line = re.sub(r'^\d+:\s*', '', lines[i].strip())
                    if is_schedule_continuation(next_line):
                        line = line + next_line
                        i += 1
                    else:
                        break
            cleaned.append(f'## {line.strip(chr(0x3000)).strip()}')
            continue
        if line.startswith('◯'):
            line = format_speaker_line(line)
            line = apply_redaction(line)
        else:
            line = line.replace('◯', '')
        line = line.replace('＿＿＿', '').strip()
        if line:
            cleaned.append(line)
    return cleaned


def make_output_stem(src_stem):
    # パターン①: 末尾が「_YYYY-MM-DD」（アンダースコア区切り）
    parts = src_stem.rsplit('_', 1)
    if len(parts) == 2 and len(parts[1]) == 10 and parts[1][4] == '-' and parts[1][7] == '-':
        date = parts[1]
        body = parts[0].replace('_本文', '').replace('本文', '').rstrip('_')
        return date + '_' + body
    # パターン②: 末尾が「 YYYY-MM-DD」（半角スペース区切り）
    parts = src_stem.rsplit(' ', 1)
    if len(parts) == 2 and len(parts[1]) == 10 and parts[1][4] == '-' and parts[1][7] == '-':
        date = parts[1]
        body = parts[0].replace('\u3000本文', '').replace(' 本文', '').replace('本文', '').rstrip()
        return date + '_' + body
    return src_stem


def convert_file(src, dst):
    content = read_file(src)
    if content is None:
        print(f'  ❌ 読み込み失敗: {src.name}')
        return False
    out_stem = make_output_stem(src.stem)
    lines    = clean_lines(content.splitlines())
    md_text  = '# ' + out_stem + '\n\n' + '\n\n'.join(lines)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(md_text, encoding='utf-8-sig')
    print(f'  ✅ {src.name}')
    print(f'      → {dst.name}')
    return True


print('✅ 関数定義完了')


In [ ]:
# セル④ 実行
out_dir = OUTPUT_DIR or INPUT_DIR
targets = sorted(
    p for p in INPUT_DIR.glob(f'*{INPUT_EXT}')
    if not p.name.startswith(OUTPUT_PREFIX)
)
if not targets:
    print(f'⚠️  対象の {INPUT_EXT} ファイルが見つかりませんでした。')
    print(f'   INPUT_DIR を確認してください: {INPUT_DIR.resolve()}')
else:
    print(f'📋 変換対象: {len(targets)} 件\n')
    success, failure = 0, 0
    for src in targets:
        dst = out_dir / (OUTPUT_PREFIX + make_output_stem(src.stem) + OUTPUT_EXT)
        if convert_file(src, dst):
            success += 1
        else:
            failure += 1
    print(f'\n{"="*40}')
    print(f'✅ 成功: {success} 件  ❌ 失敗: {failure} 件')
    print(f'出力先: {out_dir.resolve()}')


In [ ]:
# セル⑤ 変換結果のプレビュー（任意）
PREVIEW_TARGET = ''   # ← 入力ファイル名（拡張子なし）を入力
PREVIEW_LINES  = 80

if not PREVIEW_TARGET:
    print('ℹ️  PREVIEW_TARGET にファイル名（拡張子なし）を入力してください。')
else:
    out_dir = OUTPUT_DIR or INPUT_DIR
    md_path = out_dir / (OUTPUT_PREFIX + make_output_stem(PREVIEW_TARGET) + OUTPUT_EXT)
    if not md_path.exists():
        print(f'❌ ファイルが見つかりません: {md_path}')
    else:
        text    = md_path.read_text(encoding='utf-8-sig')
        preview = '\n'.join(text.splitlines()[:PREVIEW_LINES])
        display(Markdown(preview))
